<a href="https://colab.research.google.com/github/Steph-business/Tech_Talent_Accelerator/blob/main/Week8_day4_Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercices XP : MCP Minimal via STDIO (Étudiant)

Construisez un minuscule serveur et client MCP qui communiquent via STDIO. Ce code est censé être exécuté dans un notebook Jupyter local et non dans le notebook de Colab.

## Ce que vous apprendrez
- Comment le protocole MCP structure les hôtes/clients/serveurs et pourquoi STDIO est idéal localement.
- Comment enregistrer un outil (action) et une ressource (contexte en lecture seule) sur un serveur.
- Comment écrire un client qui initialise, liste et invoque ces fonctionnalités.

## Configuration
Exécutez la cellule d'installation, puis redémarrez l'environnement d'exécution si Colab le demande. Python 3.10+ est requis.

In [5]:
# Forcer l'installation de MCP et ses dépendances
import sys
!{sys.executable} -m pip install -qU "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 8.1 MB/s eta 0:00:00


In [ ]:
# Vérification rapide
!python --version
!mcp --help | head -n 5

## A. Serveur (server.py)
Créez un petit serveur MCP nommé "Demo" avec :
- Un outil `add(a: int, b: int) -> int` retournant la somme.
- Un modèle de ressource `greeting://{name}` retournant "Bonjour, {name}!".
- Démarrez la boucle STDIO dans `__main__`.

In [1]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

# Création de l'instance FastMCP nommée "Demo"
mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Retourne la somme de deux entiers."""
    # Logique : additionner a et b
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Retourne une salutation pour le nom donné."""
    # Logique : retourner la chaîne formatée en français
    return f"Bonjour, {name}!"

if __name__ == "__main__":
    # Démarrage du serveur en utilisant le transport STDIO (Standard Input/Output)
    mcp.run("stdio")

Writing server.py


## B. Client (client.py)
Écrivez un client qui :
1) Lance le serveur via STDIO en utilisant la CLI MCP.
2) Initialise une session.
3) Liste les ressources et les outils, en affichant leurs noms.
4) Lit `greeting://hello` et l'affiche.
5) Appelle l'outil `add` avec a=1, b=7 et affiche le résultat.

In [2]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Configuration des paramètres pour lancer le serveur via la commande CLI 'mcp'
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

def extract_content(payload):
    """Meilleur effort pour extraire le texte des réponses MCP."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)

async def run():
    # Connexion au serveur via le transport STDIO
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 1. Initialisation de la session
            await session.initialize()

            # 2. Lister les ressources
            resources = await session.list_resources()
            print("--- Ressources disponibles ---")
            for res in resources.resources:
                print(f"URI: {res.uri}")

            # 3. Lister les outils
            tools = await session.list_tools()
            print("\n--- Outils disponibles ---")
            for tool in tools.tools:
                print(f"Nom: {tool.name}")

            # 4. Lire la ressource greeting://hello
            greeting = await session.read_resource("greeting://hello")
            print(f"\nRésultat ressource: {extract_content(greeting)}")

            # 5. Appeler l'outil add(a=1, b=7)
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(f"Résultat outil add(1,7): {extract_content(result)}")

if __name__ == "__main__":
    # Exécution de la boucle asynchrone
    asyncio.run(run())

Writing client.py


## C. Exécution
Un terminal (le client lance le serveur) :
```
python client.py
```

Ou deux terminaux :
```
mcp run server.py
python client.py
```

Dans Colab, exécutez la cellule suivante (le client lancera automatiquement le serveur).

In [4]:
# On s'assure d'exécuter avec le binaire python actuel pour éviter les problèmes de PATH
import sys
!{sys.executable} client.py

Traceback (most recent call last):
  File "/content/client.py", line 2, in <module>
    from mcp import ClientSession, StdioServerParameters
ModuleNotFoundError: No module named 'mcp'


## Dépannage
- `mcp: command not found` ? réexécutez la cellule d'installation ou redémarrez l'environnement.
- Connexion fermée ? ouvrez un second terminal et lancez `mcp run server.py` pour vérifier les erreurs du serveur.
- Erreurs de type ? assurez-vous que les arguments JSON sont des entiers pour `add`.